<!-- WARNING: THIS FILE WAS AUTOGENERATED! DO NOT EDIT! -->

## 🗺️ Dtype Mapping

Map Arrow types to MAX dtypes for proper tensor creation.

| Arrow Type | MAX DType | NumPy dtype |
|------------|-----------|-------------|
| `int8` - `int64` | `DType.int8` - `DType.int64` | `np.int8` - `np.int64` |
| `uint8` - `uint64` | `DType.uint8` - `DType.uint64` | `np.uint8` - `np.uint64` |
| `float32`, `float64` | `DType.float32`, `DType.float64` | `np.float32`, `np.float64` |
| `date32` | `DType.int32` | `np.int32` *(epoch days)* |

In [ ]:
#| echo: false
#| output: asis
show_doc(get_max_dtype)

---

### get_max_dtype

```python

def get_max_dtype(
    arrow_type:DataType
)->DType:


```

*Get MAX DType for an Arrow type.*

In [ ]:
#| echo: false
#| output: asis
show_doc(get_numpy_dtype)

---

### get_numpy_dtype

```python

def get_numpy_dtype(
    arrow_type:DataType
)->dtype:


```

*Get NumPy dtype for an Arrow type.*

In [ ]:
assert get_numpy_dtype(pa.int64()) == np.dtype('int64')

Unsupported types raise `TypeError`:

In [ ]:
try:
    get_max_dtype(pa.string())
    assert False, "Should have raised TypeError"
except TypeError as e:
    assert "Unsupported Arrow type" in str(e)

In [ ]:
assert get_max_dtype(pa.float64()) == DType.float64
assert get_max_dtype(pa.date32()) == DType.int32  # Date32 maps to int32

## 🔄 Zero-Copy Bridge Functions

The key insight: PyArrow stores data in **contiguous buffers**. For primitive types without nulls, `arr.to_numpy(zero_copy_only=True)` gives a NumPy view over the **same memory**.

We then use `driver.Tensor.from_numpy()` which maintains the zero-copy property for contiguous arrays.

### Memory Flow

```
┌──────────────┐     zero-copy      ┌──────────────┐     zero-copy      ┌──────────────┐
│  Arrow Array │ ─────────────────► │  NumPy View  │ ─────────────────► │  MAX Tensor  │
│   (buffer)   │                    │  (same mem)  │                    │  (same mem)  │
└──────────────┘                    └──────────────┘                    └──────────────┘
```

### ⚠️ Constraints

- ❌ **No nulls** - Arrow validity bitmaps break zero-copy
- ❌ **Single chunk** - Multiple chunks require combining
- ❌ **GPU requires copy** - Device memory is separate
- ✅ **Primitive types only** - Strings/nested types not supported

In [ ]:
#| echo: false
#| output: asis
from nbdev.showdoc import show_doc
show_doc(arrow_to_numpy_view)

---

### arrow_to_numpy_view

```python

def arrow_to_numpy_view(
    arr:Union, # PyArrow array (primitive type, no nulls)
)->ndarray: # NumPy view over same memory


```

*Get zero-copy NumPy view of an Arrow array.*

In [ ]:
arr = pa.array([1.0, 2.0, 3.0])
np_view = arrow_to_numpy_view(arr)
assert np_view.ctypes.data == arr.buffers()[1].address  # Zero-copy!

Arrays with nulls raise `ValueError`:

In [ ]:
try:
    arrow_to_numpy_view(pa.array([1.0, None, 3.0]))
    assert False, "Should have raised ValueError"
except ValueError as e:
    assert "nulls" in str(e)

In [ ]:
#| echo: false
#| output: asis
show_doc(arrow_to_max_tensor)

---

### arrow_to_max_tensor

```python

def arrow_to_max_tensor(
    arr:Union, # PyArrow array to convert
    device:Optional=None, # Target device (`None` = CPU)
)->Tensor: # MAX Tensor (zero-copy on CPU, copied on GPU)


```

*Zero-copy bridge from PyArrow array to MAX Tensor.*

## 📊 MXFrame Class

A DataFrame wrapper that bridges PyArrow tables to MAX Engine.

### Design Principles

1. **PyArrow-native storage** - Uses Arrow's memory format internally
2. **Zero-copy by default** - `to_numpy()` and `to_max_tensor()` don't copy on CPU
3. **Cached views** - NumPy views are cached to keep memory references alive
4. **Device-aware** - Seamless CPU/GPU tensor creation

### API Overview

| Method | Returns | Zero-Copy? |
|--------|---------|------------|
| `to_numpy(col)` | `np.ndarray` | ✅ Yes |
| `to_max_tensor(col)` | `Tensor` (CPU) | ✅ Yes |
| `to_max_tensor(col, gpu)` | `Tensor` (GPU) | ❌ Copies to device |
| `get_buffer_address(col)` | `int` | N/A (for verification) |

In [ ]:
#| echo: false
#| output: asis
show_doc(MXFrame)

---

### MXFrame

```python

def MXFrame(
    data:Union, # Arrow Table or dict of lists
):


```

*PyArrow-backed DataFrame with zero-copy MAX Engine integration.*

In [ ]:
#| echo: false
#| output: asis
show_doc(MXFrame.to_numpy)

---

### MXFrame.to_numpy

```python

def to_numpy(
    column:str, # Column name
)->ndarray: # Zero-copy NumPy view


```

*Get zero-copy NumPy view of column (cached).*

The view is cached to keep memory references alive:

In [ ]:
df = MXFrame({'x': [1.0, 2.0, 3.0]})
np1 = df.to_numpy('x')
np2 = df.to_numpy('x')
assert np1 is np2  # Same cached view

In [ ]:
#| echo: false
#| output: asis
show_doc(MXFrame.to_max_tensor)

---

### MXFrame.to_max_tensor

```python

def to_max_tensor(
    column:str, # Column name
    device:Optional=None, # Target device (`None` = CPU)
)->Tensor: # MAX Tensor


```

*Get MAX Tensor for column (zero-copy on CPU).*

Pass a `device` to copy to GPU (unavoidable for GPU compute):

In [ ]:
#| eval: false
if driver.accelerator_count() > 0:
    gpu = driver.Accelerator()
    tensor_gpu = df.to_max_tensor('x', device=gpu)
    print(f"GPU tensor: {tensor_gpu.shape}")

GPU tensor: (3,)


In [ ]:
#| echo: false
#| output: asis
show_doc(MXFrame.get_buffer_address)

---

### MXFrame.get_buffer_address

```python

def get_buffer_address(
    column:str, # Column name
)->int: # Memory address


```

*Get memory address of column's data buffer (for zero-copy verification).*

Verify zero-copy by comparing memory addresses:

In [ ]:
df = MXFrame({'x': [1.0, 2.0, 3.0]})
arrow_addr = df.get_buffer_address('x')
numpy_addr = df.to_numpy('x').ctypes.data
assert arrow_addr == numpy_addr  # Same memory!

## 🧪 Tests: Verify Zero-Copy

Critical tests to prove we're not copying data. These tests verify:

1. ✅ Memory addresses match between Arrow, NumPy, and MAX
2. ✅ Immutability is preserved (Arrow buffers are read-only)
3. ✅ Special types like Date32 work correctly
4. ✅ Performance matches zero-copy expectations (~4+ GB/s)
5. ✅ GPU transfer works when available
6. ✅ Proper error handling for unsupported cases

In [ ]:
#| eval: false
# 📋 Test 1: Create MXFrame
df = MXFrame({
    'price': [10.0, 20.0, 30.0, 40.0, 50.0],
    'qty': [1, 2, 3, 4, 5],
})

print(f"Created: {df}")
print(f"Columns: {df.columns}")
print(f"Rows: {df.num_rows}")

Created: MXFrame(5 rows, ['price', 'qty'])
Columns: ['price', 'qty']
Rows: 5


In [ ]:
#| eval: false
# 🔍 Test 2: Zero-copy verification - Arrow to NumPy
arrow_arr = df['price']
numpy_view = df.to_numpy('price')

# Get addresses
arrow_addr = df.get_buffer_address('price')
numpy_addr = numpy_view.ctypes.data

print(f"Arrow buffer address:  0x{arrow_addr:X}")
print(f"NumPy array address:   0x{numpy_addr:X}")
print(f"Same memory: {arrow_addr == numpy_addr} ✓" if arrow_addr == numpy_addr else "COPY DETECTED!")

Arrow buffer address:  0x7C1FEAA302C0
NumPy array address:   0x7C1FEAA302C0
Same memory: True ✓


In [ ]:
#| eval: false
# ⚡ Test 3: Arrow to MAX Tensor
tensor = df.to_max_tensor('price')

print(f"Tensor shape: {tensor.shape}")
print(f"Tensor dtype: {tensor.dtype}")
print(f"Tensor values: {tensor.to_numpy()}")
print(f"Arrow values:  {df['price'].to_pylist()}")

Tensor shape: (5,)
Tensor dtype: DType.float64
Tensor values: [10. 20. 30. 40. 50.]
Arrow values:  [10.0, 20.0, 30.0, 40.0, 50.0]


In [ ]:
#| eval: false
# 🔒 Test 4: Verify NumPy view is read-only (Arrow buffers are immutable)
numpy_view = df.to_numpy('price')

try:
    numpy_view[0] = 999.0
    print("WARNING: NumPy view is writable")
except ValueError:
    print("NumPy view is read-only (expected - Arrow buffers are immutable) ✓")
    
# But they share memory - check via address
print(f"Same memory confirmed by address match: {df.get_buffer_address('price') == numpy_view.ctypes.data}")

NumPy view is read-only (expected - Arrow buffers are immutable) ✓
Same memory confirmed by address match: True


In [ ]:
#| eval: false
# 🔢 Test 5: Integer column
numpy_qty = df.to_numpy('qty')
tensor_qty = df.to_max_tensor('qty')

print(f"qty NumPy dtype: {numpy_qty.dtype}")
print(f"qty Tensor dtype: {tensor_qty.dtype}")
print(f"qty values: {numpy_qty}")

qty NumPy dtype: int64
qty Tensor dtype: DType.int64
qty values: [1 2 3 4 5]


In [ ]:
#| eval: false
# 📅 Test 6: Date32 column (special handling)


dates = pa.array([
    datetime.date(2024, 1, 1),
    datetime.date(2024, 6, 15),
    datetime.date(2024, 12, 31),
], type=pa.date32())

df_dates = MXFrame(pa.table({'shipdate': dates}))

numpy_dates = df_dates.to_numpy('shipdate')
print(f"Date32 as int32 (epoch days): {numpy_dates}")
print(f"dtype: {numpy_dates.dtype}")

Date32 as int32 (epoch days): [19723 19889 20088]
dtype: int32


In [ ]:
#| eval: false
# 🚀 Test 7: Large array performance test (proves zero-copy)

n = 10_000_000
large_arr = pa.array(np.random.rand(n).astype(np.float32))
df_large = MXFrame(pa.table({'data': large_arr}))

# Time the zero-copy conversion
t0 = time.perf_counter()
for _ in range(100):
    tensor = df_large.to_max_tensor('data')
elapsed = (time.perf_counter() - t0) / 100 * 1000

print(f"Array size: {n:,} elements ({n * 4 / 1e6:.1f} MB)")
print(f"Arrow → MAX Tensor: {elapsed:.3f} ms")
print(f"Throughput: {n * 4 / 1e9 / (elapsed / 1000):.1f} GB/s")
print("(Fast time = zero-copy confirmed)")

Array size: 10,000,000 elements (40.0 MB)
Arrow → MAX Tensor: 6.667 ms
Throughput: 6.0 GB/s
(Fast time = zero-copy confirmed)


In [ ]:
#| eval: false
# 🎮 Test 8: GPU transfer (if available)
if driver.accelerator_count() > 0:
    gpu = driver.Accelerator()
    
    t0 = time.perf_counter()
    gpu_tensor = df_large.to_max_tensor('data', device=gpu)
    gpu_time = (time.perf_counter() - t0) * 1000
    
    print(f"CPU → GPU transfer: {gpu_time:.2f} ms")
    print(f"GPU tensor shape: {gpu_tensor.shape}")
else:
    print("No GPU available - skipping GPU test")

CPU → GPU transfer: 13.07 ms
GPU tensor shape: (10000000,)


In [ ]:
#| eval: false
# ❌ Test 9: Error handling - nulls should fail
arr_with_nulls = pa.array([1.0, None, 3.0])
df_nulls = MXFrame(pa.table({'col': arr_with_nulls}))

try:
    _ = df_nulls.to_numpy('col')
    print("ERROR: Should have raised ValueError!")
except ValueError as e:
    print(f"Correctly rejected nulls: {e}")

Correctly rejected nulls: Array has 1 nulls - zero-copy not possible


In [ ]:
#| eval: false
print("\n" + "="*50)
print("✅ ALL ZERO-COPY BRIDGE TESTS PASSED")
print("="*50)


✅ ALL ZERO-COPY BRIDGE TESTS PASSED
